<a href="https://colab.research.google.com/github/javirk/europa_surface/blob/revert_fixed/DEMO_apply_LineaMapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome to the demo of LineaMapper v1.1 and 2.0. In this jupyter notebook, you will be guided through the process of retrieving predictions with LineaMapper v1.0, v1.1 and v2.0. You can upload your own geotiff image or make use of the 'Region A' from the publication "Length, width, and relative age analysis of lineaments in the Galileo regional maps with LineaMapper" (Haslebacher et al., PSJ, 2025).

Have fun, Caroline.

By default, this script retrieves predictions on a region showing the southern leading hemisphere of Jupiter's moon Europa
* for LineaMapper v1.0
   * with 224 tilesize ('geosize')
* for LineaMapper v1.1
   * with 224 tilesize
   * with 112 tilesize
* for LineaMapper v2.0
   * with 112 tilesize

# Preparation

In [ ]:
IS_COLAB = False # execute this cell if you are NOT on Google colab, but on binder

In [1]:
IS_COLAB = True # execute this cell if you ARE on Google colab

In [2]:
import os
from pathlib import Path
from datetime import datetime
import time
from osgeo import ogr

In [3]:
# if you want, you can test if you have a GPU available with this cell
import torch
torch.cuda.is_available()
# we are searching for a GPU with
# self.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# later in the LineaMapper_vX_to_img.py scripts

True

In [4]:
# The below line clones the github repository to your local or remote machine
!git clone -b revert_fixed https://github.com/javirk/europa_surface.git

Cloning into 'europa_surface'...
remote: Enumerating objects: 2121, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 2121 (delta 102), reused 66 (delta 38), pack-reused 1941 (from 2)
Receiving objects: 100% (2121/2121), 5.26 MiB | 18.07 MiB/s, done.
Resolving deltas: 100% (1690/1690), done.


In [10]:
# we need to install the geojson module
!pip install geojson

  Using cached geojson-3.2.0-py3-none-any.whl.metadata (16 kB)


We now clone the full github repository directly into Google Colab so that we can access every script. We change directory so that we are inside the cloned repository.

In [7]:
# we access the repository to import modules below
# also ONLY IF NOT ON BINDER:
if IS_COLAB == True:
    %cd europa_surface
    # and we need to install gdal in google Colab (following https://stackoverflow.com/questions/70275565/how-to-install-gdal-on-google-colab-fast)
    # this is for command line tools, which we'll use later
    !apt install gdal-bin


/content/europa_surface
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  python3-gdal python3-numpy
Suggested packages:
  libgdal-grass python-numpy-doc python3-pytest
The following NEW packages will be installed:
  gdal-bin python3-gdal python3-numpy
0 upgraded, 3 newly installed, 0 to remove and 34 not upgraded.
Need to get 5,055 kB of archives.
After this operation, 25.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-numpy amd64 1:1.21.5-1ubuntu22.04.1 [3,467 kB]
Err:2 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 python3-gdal amd64 3.6.4+dfsg-1~jammy0
  404  Not Found [IP: 185.125.190.80 443]
Err:3 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 gdal-bin amd64 3.6.4+dfsg-1~jammy0
  404  Not Found [IP: 185.125.190.80 443]
Fetched 3,467 kB in 2s (1,402 kB/s)
E: Failed t

In [8]:
!pip install -r requirements_full.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.1 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 3.0.0 Requires-Python >=3.7, <=3.11
ERROR: Could not find a version that satisfies the requirement pywin32==310 (from versions: none)
ERROR: No matching distribution found for pywin32==310


If you want to run LineaMapper on your own geotiff, simply put/upload your geotiff to the folder './demo/v1_1' (and remove everything for which you do not want predictions). Here, we run LineaMapper on the geotiff in './demo/v1_1' of region A from the publication.

In [29]:
# define the input path
# note: basepath is used to define the savepath
basepath = Path('.')

# we get this from the github repository
source_path = basepath / './demo/v1_1'
# because the next line catches every file that is ending in '.tif', you can also upload your own geotiff file here
tifpaths = sorted(source_path.glob('*.tif'))

dt_string = datetime.now().strftime("%Y_%m_%d")

Next, we download the weights for LineaMapper version 1.0, 1.1 and 2.0 and put them into a subdirectory './ckpts' (checkpoints). If for any reason, the direct download form Mendeley fails (because the links were put in before they were activated), go to the Mendeley and download the weights manually and put them into a sub-directory called 'ckpts' in the directory where this notebook is located. The link to Mendeley is likely: https://data.mendeley.com/datasets/rjhsjrnxgv/1
You can also go to https://github.com/javirk/europa_surface and download the latest code. The weights correspond to versions as follows:
* LM1.0: Mask_R-CNN_pub2_run23_end_model.pt
* LM1.1: Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt
* LM2.0: bbox_vit_b_final.pt
* other files are other versions of LM2.0 (see paper)

In [11]:
import requests

# LineaMapper v1.0 weights: from Mendeley data repo (these work for sure)
# url = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"

url_LM1_0 = "https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/1815514f-a8b9-40cc-9822-1a3e41fc56d0/file_downloaded"
url_LM1_1 = "https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/5b40d00e-dfdd-4803-857a-e87815988992/file_downloaded"
url_LM2_0 = "https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/9d4f74fe-b794-4388-8c5e-416468277531/file_downloaded"

# # for testing, before Mendeley Data repo is published:
# url_LM1_0 = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"
# url_LM1_1 = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"
# url_LM2_0 = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"

for url, LMname in [(url_LM1_0, "Mask_R-CNN_pub2_run23_end_model.pt"), (url_LM1_1, "Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt"), (url_LM2_0, "bbox_vit_b_final.pt")]:
    print(url, LMname)
    # Download the file
    response = requests.get(url)
    response.raise_for_status()  # Ensure the request was successful

    # Save the file locally, in a subfolder called 'ckpts' (for checkpoints)
    %mkdir ckpts
    with open(f"./ckpts/{LMname}", "wb") as file:
        file.write(response.content)



https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/1815514f-a8b9-40cc-9822-1a3e41fc56d0/file_downloaded Mask_R-CNN_pub2_run23_end_model.pt
https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/5b40d00e-dfdd-4803-857a-e87815988992/file_downloaded Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt
mkdir: cannot create directory ‘ckpts’: File exists
https://data.mendeley.com/public-files/datasets/rjhsjrnxgv/files/9d4f74fe-b794-4388-8c5e-416468277531/file_downloaded bbox_vit_b_final.pt
mkdir: cannot create directory ‘ckpts’: File exists


Below, we define a convenient straightforward function to execute LineaMapper_v1_to_img.py or LineaMapper_v2_to_img.py that we can call later for different versions. We are preparing a command-line that calls the script and passes on arguments. You can change these arguments and see what effect they have. Very briefly:


* geofile: full path with filename in TIFF format. The full path of the file for which predictions are seeked. Must be a GEOTIFF file.
* savedir: path to directory where the output is stored. there will be subdirectories automatically generated by the function.
* mask_threshold: threshold for float mask. The float mask that is output by the model gets converted to a binary mask using this threshold. Default is 0.5.
* iou_threshold: threshold for Intersection-Over-Union (IoU). The computed mask IoU is multiplied with the multiplication factor and then tested against the IoU threshold. Default is 0.5.
* multiplication_factor: factor by which computed mask Intersection-Over-Union (IoU) gets multiplied. This compensates for the moving window algorithm.
* del_pxs: If a boolean mask has an area lower than del_pxs, it gets deleted.
* class_scores: list with score thresholds for individual classes. If not given, defaults to 0.5 for each class. Classes are 1) bands 2) double ridges 3) ridge complexes 4) undifferentiated lineae.
* geosize: Choose a tile size that is fed to the network. This tile gets re-cast to 200x200 or 300x300. (minsize, maxsize)
* cut_size: Choose a size for cutting the input image into subimages. The image gets tiled up into smaller subimages, if it is bigger than cut_size.
* azimuth_diff_range: Choose a maximal difference between two azimuths so that they are still considered the same direction. Masks that fulfill the IoU criterion, but are not going into the same direction, are not merged.
* modelname: path to .pt file of the model.
* minsize/maxsize: Choose the minsize/maxsize parameter for the Mask R-CNN. This tile gets re-cast to (minsize, maxsize)
* sampath: .pt file of the SAM model, in the model_dict subdirectory.
* sam_modus: model architecture. vit_b or vit_t

You can explore all arguments in LineaMapper_v1_to_img.py.



In [32]:
import subprocess
# we define this routine only to get debugging help if anything goes wrong
def run_command(command):
    try:
        # Run the command
        result = subprocess.run(command, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        # Capture standard output
        stdout = result.stdout
        # Capture standard error
        stderr = result.stderr

        # Print helpful debugging information
        print("Command executed successfully.")
        print("Standard Output:")
        print(stdout)

        if stderr:
            print("Standard Error:")
            print(stderr)

    except subprocess.CalledProcessError as e:
        # Capture error output in case of an error
        print("An error occurred while executing the command.")
        print("Standard Output:")
        print(e.stdout)
        print("Standard Error:")
        print(e.stderr)

    return


In [36]:
# We define a straightforward function to execute LineaMapper_v1.py or ..._v2.py that we can call later for different models
# this function automatically loops through all found tiff files
# it stores the output as a geojson and a shapefile, along with a text file with the parameters we used
def forward_LM(modelname, version, subset, geosize):
    # NOTE: the savepath does not yet need to exist
    savepath = basepath / 'LineaMapper_output' / (dt_string + '_RegionA_' + subset)
    # start full time
    full_time_start = time.time()

    for tifffile in tifpaths:
        tf = tifffile.stem
        print(tf)
        command = f'python LineaMapper_{version}_to_img.py --modelname={modelname} --geofile=' + str(source_path.joinpath(tf + '.tif')) + ' --savedir=' + str(savepath) + f' --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize={geosize}'
        print(command)
        # os.system(command)
        # print(os.popen(command).read()) # for jupyter notebook, we need this line to execute the command
        run_command(command)

    # measure time and simply print
    timesum = time.time() - full_time_start
    print('this script took {:.2f} seconds to execute. Makes {:.2f} hours.'.format(timesum, timesum/3600))
    # simple test:
    # python LineaMapper_to_img.py --geofile=z:/Groups/PIG/Caroline/isis/data/galileo/usgs_photogrammetrically/Europa_Mosaics_Equirectangular/E6ESCRATER01_GalileoSSI_Equi-cog.tif --savedir=z:/Groups/PIG/Caroline/isis/data/galileo/usgs_photogrammetrically/LineaMapper_output/tests

    ######## convert to shapefiles
    # problem is that I do not have the exact filename, so I retrieve it simply afterwards
    # make shape_file folder
    os.makedirs(Path(savepath) / 'shape_files', exist_ok=True)

    geojfiles = sorted((Path(savepath) / 'json_files').glob('*.geojson'))
    for geojfile in geojfiles:
        # convert to shapefile as well
        command = 'ogr2ogr -nlt POLYGON -skipfailures {} {}'.format((savepath / 'shape_files').joinpath(geojfile.stem + '.shp'), (savepath / 'json_files').joinpath(geojfile.stem + '.geojson'))
        print(command)
        # os.system(command)
        # print(os.popen(command).read()) # for jupyter notebook, we need this line to execute the command
        run_command(command)

    # check if it worked for all
    geoshpfiles = sorted((Path(savepath) / 'shape_files').glob('*.shp'))

    if len(geoshpfiles) == len(geojfiles):
        print('ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.')
    else:
        raise Warning('some geojson files lead to errors, it seems.')

    return

# Running LineaMapper

Now, we are already prepared to actually run LineaMapper on our input geotiff image(s). This can take up to 1 hour on a Google Colab CPU. Depending what computing power you have available, it can be much faster. In google colab, you can start up a GPU by going to 'runtime' > Change runtime type, and select a GPU. However, we have noticed that running the external script does not automatically recognise the GPU. If this happens to you, you might want to crop the geotiff input image for faster results.
Else, you can run the next cell to select a cropped version (800x800 px) of region A. This should take 4 minutes for one model.

In [33]:
# OPTIONAL: adapt tifpaths to take smaller image (for faster inference)
source_path = basepath / './demo/v1_1/800px' # we also need to adapt the source path
tifpaths = sorted(source_path.glob('*.tif'))

In [34]:
tifpaths

[PosixPath('demo/v1_1/800px/17ESREGMAP02_Bland2021_800px.tif')]

In [37]:
# for LineaMapper v1.0
#    on 224 geosize
subset = 'LM1.0_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Mask_R-CNN_pub2_run23_end_model.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

17ESREGMAP02_Bland2021_800px
python LineaMapper_v1_to_img.py --modelname=./ckpts/Mask_R-CNN_pub2_run23_end_model.pt --geofile=demo/v1_1/800px/17ESREGMAP02_Bland2021_800px.tif --savedir=LineaMapper_output/2025_05_19_RegionA_LM1.0_224 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize=224
this script took 0.00 seconds to execute. Makes 0.00 hours.
ogr2ogr -nlt POLYGON -skipfailures LineaMapper_output/2025_05_19_RegionA_LM1.0_224/shape_files/2025_05_19_22_33_17ESREGMAP02_Bland2021_800px_0.shp LineaMapper_output/2025_05_19_RegionA_LM1.0_224/json_files/2025_05_19_22_33_17ESREGMAP02_Bland2021_800px_0.geojson
An error occurred while executing the command.
Standard Output:

Standard Error:
/bin/sh: 1: ogr2ogr: not found



Warning: some geojson files lead to errors, it seems.

In [ ]:
# for LineaMapper v1.1
#     on 224 geosize (tiles)
subset = 'LM1.1_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)
#     on 112 geosize
geosize = 112
subset = 'LM1.1_112'
forward_LM(modelname, version, subset, geosize)

17ESREGMAP02_Bland2021_regionB
python LineaMapper_v1_to_img.py --modelname=./ckpts/Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt --geofile=demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif --savedir=LineaMapper_output/2025_04_23_RegionA_LM1.1_224 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize=224
17ESREGMAP02_Bland2021_regionB
class scores: {1: np.float64(0.5), 2: np.float64(0.5), 3: np.float64(0.5), 4: np.float64(0.5)}
demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif
minsize: 200, maxsize: 300

this script took 6.74 seconds to execute. Makes 0.00 hours.
ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.
17ESREGMAP02_Bland2021_regionB
python LineaMapper_v1_to_img.py --modelname=./ckpts/Mask_R-CNN_v1_1_17ESREGMAP02_part01_run10_end_model.pt --geofile=demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif --savedir=LineaMapper_output/2025_04_23_RegionA_LM1.1_112 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_f

In [ ]:
# for LineaMapper v2.0
#     on 112 geosize
# note that sampath and sammodus are the default. "./ckpts/bbox_vit_b_final.pt", 'vit_b'
subset = 'LM2.0_112' # identification string for savepath
geosize = 112
modelname = './ckpts/bbox_vit_b_final.pt'
version = 'v2' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

Now, you can download the output from generated folder "LineaMapper_output", pull them into a GIS application (such as open source QGIS), and inspect and compare the output. If you have used the demo image, you should also see a preview pdf image. (Please note that the preview feature is only working for images smaller than the cut size, because the display_preview routine is not currently updated for the use of subimages.)

If the Mendeley data download fails, try this:

In [ ]:
import requests

# LineaMapper v1.0 weights: from Mendeley data repo for Haslebacher et al. (2024)
url = "https://data.mendeley.com/public-files/datasets/nxnwj6hy4s/files/b248180d-aacb-43b4-a541-0b60d8076a3e/file_downloaded"

# Download the file
response = requests.get(url)
response.raise_for_status()  # Ensure the request was successful

# Save the file locally, in a subfolder called 'ckpts' (for checkpoints)
%mkdir ckpts
with open("./ckpts/Weights_v1_0.pt", "wb") as file:
    file.write(response.content)

mkdir: cannot create directory ‘ckpts’: File exists


In [ ]:
# for LineaMapper v1.0
#    on 224 geosize
subset = 'LM1.0_224' # identification string for savepath
geosize = 224
modelname = './ckpts/Weights_v1_0.pt'
version = 'v1' # 'v1' or 'v2'
# main
forward_LM(modelname, version, subset, geosize)

17ESREGMAP02_Bland2021_regionB
python LineaMapper_v1_to_img.py --modelname=./ckpts/Mask_R-CNN_pub2_run23_end_model.pt --geofile=demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif --savedir=LineaMapper_output/2025_04_24_RegionA_LM1.0_224 --class_scores 0.5 0.5 0.5 0.5 --cut_size=3000 --multiplication_factor=15 --azimuth_diff_range=25 --del_pxs=100 --geosize=224
17ESREGMAP02_Bland2021_regionB
class scores: {1: np.float64(0.5), 2: np.float64(0.5), 3: np.float64(0.5), 4: np.float64(0.5)}
demo/v1_1/17ESREGMAP02_Bland2021_regionB.tif
minsize: 200, maxsize: 300

this script took 7.25 seconds to execute. Makes 0.00 hours.
ALL GEOJSON FILES WERE CONVERTED TO SHAPEFILES.
